# Notebook 1 — Univariate & Bivariate EDA (Interactive Plotly)

This notebook produces comprehensive interactive visualizations for the Telco dataset:
- **Univariate**: Senior citizen distribution, MonthlyCharges density, Tenure distribution
- **Bivariate & Correlation**: Churn drivers by Contract, PaymentMethod, InternetService; numerical correlations; mutual information scores

In [ ]:
# Imports and data load
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_PATH = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
assert os.path.exists(DATA_PATH), f"Dataset not found at {DATA_PATH}"

# Load dataset and do minimal cleaning used by these plots
df = pd.read_csv(DATA_PATH, low_memory=False)

# Ensure numeric columns are typed correctly (guard against whitespace TotalCharges)
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip().replace('', np.nan), errors='coerce')
if 'MonthlyCharges' in df.columns:
    df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
if 'tenure' in df.columns:
    df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce').fillna(0).astype(int)

# Create readable Senior flag
if 'SeniorCitizen' in df.columns:
    df['SeniorFlag'] = df['SeniorCitizen'].apply(lambda x: 'Senior' if int(x) == 1 else 'Non-Senior')
else:
    df['SeniorFlag'] = 'Unknown'

print('Loaded rows:', len(df))
print('Columns:', df.columns.tolist())

## UNIVARIATE ANALYSIS

In [ ]:
# Plot 1: Demographics — SeniorCitizen split (bar chart)
fig = px.bar(
    df['SeniorFlag'].value_counts().reset_index().rename(columns={'index':'SeniorFlag', 'SeniorFlag':'count'}),
    x='SeniorFlag',
    y='count',
    color='SeniorFlag',
    color_discrete_map={'Senior':'#2a9d8f', 'Non-Senior':'#264653', 'Unknown':'#e9c46a'},
    text='count',
    labels={'SeniorFlag':'Senior status','count':'Number of customers'},
    title='Demographics — Senior Citizen Split',
    template='plotly_white'
)
fig.update_traces(marker_line_width=0.4, hovertemplate='%{x}: %{y} customers')
fig.update_layout(
    title=dict(x=0.5, xanchor='center', font=dict(size=18)),
    xaxis_title='Senior Status',
    yaxis_title='Customer Count',
    showlegend=False,
    font=dict(size=12)
)
fig.show()

Check senior vs non-senior representation and note any imbalance impacting retention prioritization.
If seniors are overrepresented, consider dedicated support/offer strategies for this segment.

In [ ]:
# Plot 2: Financials — MonthlyCharges histogram + KDE overlay
x = df['MonthlyCharges'].dropna().values
if len(x) == 0:
    raise ValueError('No MonthlyCharges data available')

# Base histogram (density normalized) using Plotly Express for interactive hover
hist = px.histogram(
    df,
    x='MonthlyCharges',
    nbins=60,
    histnorm='density',
    opacity=0.6,
    color_discrete_sequence=['#1170aa'],
    labels={'MonthlyCharges':'Monthly Charges ($)','count':'Density'},
    title='Financials — MonthlyCharges Density with KDE',
    template='plotly_white'
)
hist.update_traces(hovertemplate='MonthlyCharges: %{x}<br>Density: %{y:.4f}')

# Attempt KDE with scipy, fallback to simple smoothed histogram density if scipy unavailable
try:
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(x)
    xx = np.linspace(x.min(), x.max(), 300)
    yy = kde(xx)
    kde_trace = go.Scatter(x=xx, y=yy, mode='lines', name='KDE', line=dict(color='#ef476f', width=2))
except Exception:
    # Fallback: smooth the histogram density
    counts, bins = np.histogram(x, bins=60, density=True)
    centers = 0.5 * (bins[:-1] + bins[1:])
    # simple smoothing
    yy = np.convolve(counts, np.ones(5)/5, mode='same')
    kde_trace = go.Scatter(x=centers, y=yy, mode='lines', name='ApproxDensity', line=dict(color='#ef476f', width=2, dash='dash'))

fig = go.Figure(data=hist.data + (kde_trace,))
fig.update_layout(
    title=dict(text='Financials — MonthlyCharges Density with KDE', x=0.5, xanchor='center', font=dict(size=18)),
    xaxis_title='Monthly Charges (USD)',
    yaxis_title='Density',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white',
    font=dict(size=12)
)
fig.show()

Examine billing density tails and common billing clusters to inform pricing/offer segmentation.
Flag high-density regions for A/B tests on pricing sensitivity and discounting strategies.

In [ ]:
# Plot 3: Lifecycles — Tenure histogram
if 'tenure' not in df.columns:
    raise ValueError('`tenure` column missing from dataset')

fig = px.histogram(
    df,
    x='tenure',
    nbins=60,
    color_discrete_sequence=['#8a2be2'],
    labels={'tenure':'Tenure (months)','count':'Number of customers'},
    title='Lifecycles — Tenure Distribution (Account Length)',
    template='plotly_white'
)
fig.update_traces(marker_line_width=0.4, hovertemplate='Tenure: %{x} months<br>Count: %{y}')
fig.update_layout(
    title=dict(text='Lifecycles — Tenure Distribution (Account Length)', x=0.5, xanchor='center', font=dict(size=18)),
    xaxis_title='Tenure (months)',
    yaxis_title='Number of customers',
    font=dict(size=12)
)
fig.show()

Identify tenure cohorts (e.g., 0–12, 13–24, 25+) and prioritize retention for high-churn cohorts.
Use tenure peaks to guide onboarding improvements and lifecycle-tailored campaigns.

## BIVARIATE & CORRELATION ANALYSIS

In [ ]:
# Plot 1: Normalized stacked horizontal bar — Churn across Contract types
if 'Contract' in df.columns and 'Churn' in df.columns:
    churn_contract = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
    churn_contract = churn_contract.reset_index()
    churn_contract_melted = churn_contract.melt(id_vars='Contract', var_name='Churn', value_name='Percentage')
    
    fig = px.bar(
        churn_contract_melted,
        y='Contract',
        x='Percentage',
        color='Churn',
        color_discrete_map={'Yes':'#d62728', 'No':'#2ca02c'},
        barmode='stack',
        labels={'Percentage':'Percentage (%)', 'Churn':'Churn Status'},
        title='Churn Variance Across Contract Types (Normalized)',
        template='plotly_white'
    )
    fig.update_layout(
        title=dict(x=0.5, xanchor='center', font=dict(size=18)),
        xaxis_title='Churn Percentage (%)',
        yaxis_title='Contract Type',
        legend=dict(title='Churn Status', orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.02),
        font=dict(size=12)
    )
    fig.update_traces(hovertemplate='%{y}<br>%{customdata}<br>%{x:.2f}%', customdata=churn_contract_melted['Churn'])
    fig.show()
else:
    print('Contract or Churn column not found.')

Month-to-month contracts show significantly higher churn; consider proactive upgrade campaigns.
Two-year contracts demonstrate strong retention—incentivize conversion from month-to-month plans.

In [ ]:
# Plot 2: Grouped bar chart — Churn distributions across PaymentMethod
if 'PaymentMethod' in df.columns and 'Churn' in df.columns:
    churn_payment = df.groupby(['PaymentMethod', 'Churn']).size().reset_index(name='Count')
    
    fig = px.bar(
        churn_payment,
        x='PaymentMethod',
        y='Count',
        color='Churn',
        color_discrete_map={'Yes':'#ff7f0e', 'No':'#1f77b4'},
        barmode='group',
        labels={'Count':'Number of Customers', 'Churn':'Churn Status'},
        title='Churn Distribution Across Payment Methods',
        template='plotly_white'
    )
    fig.update_layout(
        title=dict(x=0.5, xanchor='center', font=dict(size=18)),
        xaxis_title='Payment Method',
        yaxis_title='Number of Customers',
        xaxis_tickangle=-45,
        legend=dict(title='Churn Status', orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        font=dict(size=12)
    )
    fig.update_traces(hovertemplate='%{x}<br>Churn: %{customdata}<br>Count: %{y}', customdata=churn_payment['Churn'])
    fig.show()
else:
    print('PaymentMethod or Churn column not found.')

Identify payment methods with elevated churn (e.g., paper checks) and promote digital automated alternatives.
Analyze friction in payment experience—electronic/automatic methods may reduce churn through convenience.

In [ ]:
# Plot 3: Segmentation breakdown — Churn across InternetService classes
if 'InternetService' in df.columns and 'Churn' in df.columns:
    churn_internet = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
    churn_internet = churn_internet.reset_index()
    churn_internet_melted = churn_internet.melt(id_vars='InternetService', var_name='Churn', value_name='Percentage')
    
    fig = px.bar(
        churn_internet_melted,
        x='InternetService',
        y='Percentage',
        color='Churn',
        color_discrete_map={'Yes':'#e74c3c', 'No':'#27ae60'},
        barmode='stack',
        labels={'Percentage':'Churn Percentage (%)', 'Churn':'Churn Status'},
        title='Customer Drop-off Paths Across Internet Service Classes',
        template='plotly_white',
        text='Percentage'
    )
    fig.update_traces(
        textposition='inside',
        texttemplate='%{text:.1f}%',
        hovertemplate='%{x}<br>%{customdata}<br>%{y:.2f}%',
        customdata=churn_internet_melted['Churn']
    )
    fig.update_layout(
        title=dict(x=0.5, xanchor='center', font=dict(size=18)),
        xaxis_title='Internet Service Type',
        yaxis_title='Percentage (%)',
        legend=dict(title='Churn Status', orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        font=dict(size=12)
    )
    fig.show()
else:
    print('InternetService or Churn column not found.')

Fiber optic customers may show high churn if service quality issues exist—prioritize network diagnostics.
DSL and No-internet segments show different dynamics; tailor retention tactics by service type and quality SLAs.

In [ ]:
# Plot 4: Correlation Matrix (Pearson) — Numerical features
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
numerical_data = df[numerical_features].dropna()

if len(numerical_data) > 0:
    correlation_matrix = numerical_data.corr(method='pearson')
    
    fig = go.Figure(data=go.Heatmap(
        z=correlation_matrix.values,
        x=correlation_matrix.columns,
        y=correlation_matrix.columns,
        colorscale='RdBu',
        zmid=0,
        text=np.round(correlation_matrix.values, 3),
        texttemplate='%{text:.3f}',
        textfont={'size': 12},
        colorbar=dict(title='Pearson r', tickformat='.2f'),
        hovertemplate='%{y} vs %{x}<br>r = %{z:.4f}<extra></extra>'
    ))
    fig.update_layout(
        title=dict(text='Correlation Matrix — Numerical Features (Pearson r)', x=0.5, xanchor='center', font=dict(size=18)),
        xaxis_title='Features',
        yaxis_title='Features',
        template='plotly_white',
        font=dict(size=12),
        width=700,
        height=600
    )
    fig.show()
    print('Correlation Matrix:')
    print(correlation_matrix)
else:
    print('Insufficient numerical data for correlation analysis.')

High TotalCharges–MonthlyCharges correlation (>0.8) suggests strong multicollinearity; consider feature engineering.
Tenure shows weak-to-moderate correlations with billing features; evaluate as independent predictor for churn modeling.

In [ ]:
# Plot 5: Mutual Information scores — Categorical predictors vs Churn
try:
    from sklearn.feature_selection import mutual_info_classif
    
    # Select categorical columns (exclude target)
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    if 'Churn' in categorical_cols:
        categorical_cols.remove('Churn')
    
    if len(categorical_cols) > 0 and 'Churn' in df.columns:
        # Encode categorical features as integers
        df_encoded = df[categorical_cols].copy()
        for col in categorical_cols:
            df_encoded[col] = pd.factorize(df_encoded[col])[0]
        
        # Encode target
        y = pd.factorize(df['Churn'])[0]
        
        # Compute MI scores
        mi_scores = mutual_info_classif(df_encoded, y, random_state=42)
        mi_df = pd.DataFrame({'Feature': categorical_cols, 'MI_Score': mi_scores}).sort_values('MI_Score', ascending=True)
        
        fig = px.bar(
            mi_df,
            x='MI_Score',
            y='Feature',
            color='MI_Score',
            color_continuous_scale='Viridis',
            labels={'MI_Score':'Mutual Information Score'},
            title='Mutual Information Scores — Categorical Predictors vs Churn',
            template='plotly_white',
            orientation='h'
        )
        fig.update_layout(
            title=dict(x=0.5, xanchor='center', font=dict(size=18)),
            xaxis_title='Mutual Information Score',
            yaxis_title='Categorical Feature',
            showlegend=False,
            font=dict(size=12)
        )
        fig.update_traces(hovertemplate='%{y}<br>MI Score: %{x:.4f}')
        fig.show()
        print('\nMutual Information Scores (sorted):')
        print(mi_df.sort_values('MI_Score', ascending=False))
    else:
        print('No categorical features found or Churn column missing.')
except ImportError:
    print('scikit-learn not available; MI computation skipped. Install with: pip install scikit-learn')

Features with high MI scores (e.g., Contract, InternetService) are strong churn predictors; prioritize in feature selection.
Low MI features may be candidates for removal to reduce model complexity; validate domain importance before exclusion.